# Bedrock (知识库) 检索器

本指南将帮助您开始使用 AWS 知识库 [检索器](/docs/concepts/retrievers)。

[Amazon Bedrock 知识库](https://aws.amazon.com/bedrock/knowledge-bases/) 是亚马逊网络服务 (AWS) 的一项服务，它允许您通过使用您的私有数据来定制 FM 响应，从而快速构建 RAG 应用程序。

实现 `RAG` 需要组织执行多个繁琐的步骤，将数据转换为嵌入（向量），将嵌入存储在专门的向量数据库中，并构建到数据库的自定义集成，以搜索和检索与用户查询相关的文本。这可能既耗时又效率低下。

通过 `Amazon Bedrock 知识库`，您只需指向您在 `Amazon S3` 中的数据位置，`Amazon Bedrock 知识库` 即可处理整个数据摄取工作流到您的向量数据库。如果您没有现有的向量数据库，Amazon Bedrock 会为您创建一个 Amazon OpenSearch Serverless 向量存储。对于检索，请通过检索 API 使用 Langchain - Amazon Bedrock 集成，从知识库中检索与用户查询相关的结果。

### 集成详情

import {ItemTable} from "@theme/FeatureTables";

<ItemTable category="document_retrievers" item="AmazonKnowledgeBasesRetriever" />

## 设置

知识库可以通过 [AWS Console](https://aws.amazon.com/console/) 或使用 [AWS SDKs](https://aws.amazon.com/developer/tools/) 进行配置。我们需要 `knowledge_base_id` 来实例化检索器。

如果您想从单个查询中获取自动化跟踪，您还可以通过取消注释以下内容来设置您的 [LangSmith](https://docs.smith.langchain.com/) API 密钥：

In [ ]:
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")
# os.environ["LANGSMITH_TRACING"] = "true"

### 安装

此检索器位于 `langchain-aws` 包中：

In [ ]:
%pip install -qU langchain-aws

## 实例化

现在我们可以实例化我们的检索器了：

In [1]:
from langchain_aws.retrievers import AmazonKnowledgeBasesRetriever

retriever = AmazonKnowledgeBasesRetriever(
    knowledge_base_id="PUIJP4EQUA",
    retrieval_config={"vectorSearchConfiguration": {"numberOfResults": 4}},
)

## 用法

In [ ]:
query = "What did the president say about Ketanji Brown?"

retriever.invoke(query)

## 在链中使用

You can use `use` within a chain of hooks.

```jsx
import { use, useEffect } from 'react';

functionExpensiveResource() {
  // Simulate loading a resource
  return { data: 'some data' };
}

function MyComponent({ resource }) {
  const { data } = use(resource);

  useEffect(() => {
    console.log('Resource loaded:', data);
  }, [data]);

  return <div>Data: {data}</div>;
}

function App() {
  const resource = ExpensiveResource();

  return <MyComponent resource={resource} />;
}
```

In this example, `ExpensiveResource()` is called, and its result is passed to `use()`. React will suspend until the `resource` is ready, and then `MyComponent` will render with the resolved `data`.

You can also use `use` within a chain of other hooks.

```jsx
import { use, useEffect } from 'react';

function useResource() {
  // Simulate loading a resource
  return { data: 'some data' };
}

function useData() {
  const resource = useResource();
  const { data } = use(resource);
  return data;
}

function MyComponent() {
  const data = useData();

  useEffect(() => {
    console.log('Data loaded:', data);
  }, [data]);

  return <div>Data: {data}</div>;
}
```

In this example, `useResource()` is called within `useData()`, and its result is passed to `use()`. React will suspend until the `resource` is ready, and then `useData()` will return the resolved `data`. `MyComponent` will then render with the data.

**Important:**

*   `use` can only be called within a React component or another hook.
*   `use` cannot be called inside event handlers, lifecycle methods, or other non-hook functions.

You can also use `use` with Promises.

```jsx
import { use, useEffect } from 'react';

function fetchData() {
  return new Promise(resolve => {
    setTimeout(() => {
      resolve({ data: 'fetched data' });
    }, 1000);
  });
}

function MyComponent() {
  const { data } = use(fetchData());

  useEffect(() => {
    console.log('Data fetched:', data);
  }, [data]);

  return <div>Data: {data}</div>;
}
```

In this example, `fetchData()` returns a Promise. React will suspend until the Promise resolves, and then `MyComponent` will render with the resolved `data`.

You can also use `use` with objects that have a `then` method (like Promises).

```jsx
import { use, useEffect } from 'react';

const resource = {
  then(resolve) {
    setTimeout(() => {
      resolve({ data: 'resource data' });
    }, 1000);
  }
};

function MyComponent() {
  const { data } = use(resource);

  useEffect(() => {
    console.log('Resource loaded:', data);
  }, [data]);

  return <div>Data: {data}</div>;
}
```

In this example, `resource` is an object with a `then` method. React will call the `then` method and suspend until it resolves, and then `MyComponent` will render with the resolved `data`.

You can also use `use` with arrays of Promises or objects with `then` methods.

```jsx
import { use, useEffect } from 'react';

const promise1 = new Promise(resolve => setTimeout(() => resolve('data1'), 1000));
const promise2 = new Promise(resolve => setTimeout(() => resolve('data2'), 2000));

const resourceArray = [promise1, promise2];

function MyComponent() {
  const [data1, data2] = use(resourceArray);

  useEffect(() => {
    console.log('Data loaded:', data1, data2);
  }, [data1, data2]);

  return (
    <div>
      Data 1: {data1}
      <br />
      Data 2: {data2}
    </div>
  );
}
```

In this example, `resourceArray` is an array of Promises. React will suspend until all Promises resolve, and then `MyComponent` will render with the resolved data.

You can also use `use` with objects where values are Promises or objects with `then` methods.

```jsx
import { use, useEffect } from 'react';

const promise1 = new Promise(resolve => setTimeout(() => resolve('data1'), 1000));
const promise2 = new Promise(resolve => setTimeout(() => resolve('data2'), 2000));

const resourceObject = {
  data1: promise1,
  data2: promise2,
};

function MyComponent() {
  const { data1, data2 } = use(resourceObject);

  useEffect(() => {
    console.log('Data loaded:', data1, data2);
  }, [data1, data2]);

  return (
    <div>
      Data 1: {data1}
      <br />
      Data 2: {data2}
    </div>
  );
}
```

In this example, `resourceObject` is an object where values are Promises. React will suspend until all Promises resolve, and then `MyComponent` will render with the resolved data.

You can also use `use` with a mix of Promises and regular values.

```jsx
import { use, useEffect } from 'react';

const promise1 = new Promise(resolve => setTimeout(() => resolve('data1'), 1000));
const regularValue = 'regular data';

const mixedResource = [promise1, regularValue];

function MyComponent() {
  const [data1, data2] = use(mixedResource);

  useEffect(() => {
    console.log('Data loaded:', data1, data2);
  }, [data1, data2]);

  return (
    <div>
      Data 1: {data1}
      <br />
      Data 2: {data2}
    </div>
  );
}
```

In this example, `mixedResource` is an array with a Promise and a regular value. React will suspend until the Promise resolves, and then `MyComponent` will render with the resolved data.

You can also use `use` with nested Promises or objects.

```jsx
import { use, useEffect } from 'react';

const nestedPromise = new Promise(resolve =>
  setTimeout(() => resolve({ message: 'nested data' }), 1000)
);

const resource = {
  data: nestedPromise,
};

function MyComponent() {
  const { data } = use(resource);

  useEffect(() => {
    console.log('Data loaded:', data);
  }, [data]);

  return <div>Data: {data.message}</div>;
}
```

In this example, `resource` is an object with a nested Promise. React will suspend until the nested Promise resolves, and then `MyComponent` will render with the resolved data.

**Error Handling:**

If the Promise passed to `use` rejects, React will throw an error. You can catch these errors using `ErrorBoundary` components.

```jsx
import { use, useEffect } from 'react';

function fetchWithError() {
  return new Promise((_, reject) => {
    setTimeout(() => {
      reject(new Error('Failed to fetch'));
    }, 1000);
  });
}

function ErrorBoundary({ children }) {
  // Error boundary logic here
  return children;
}

function MyComponent() {
  const { data } = use(fetchWithError());

  return <div>Data: {data}</div>;
}

function App() {
  return (
    <ErrorBoundary>
      <MyComponent />
    </ErrorBoundary>
  );
}
```

In this example, `fetchWithError()` rejects. The `ErrorBoundary` component will catch the error and display an error message.

**Key Concepts:**

*   **Suspense:** `use` works in conjunction with React Suspense. When `use` is called with a Promise or an object with a `then` method, React will suspend rendering until the Promise resolves or the `then` method completes.
*   **Data Fetching:** `use` is primarily intended for data fetching, but it can be used with any asynchronous operation that returns a Promise or an object with a `then` method.
*   **Composability:** `use` is highly composable, allowing you to chain multiple asynchronous operations together and handle them seamlessly within your React components.

In [ ]:
from botocore.client import Config
from langchain.chains import RetrievalQA
from langchain_aws import Bedrock

model_kwargs_claude = {"temperature": 0, "top_k": 10, "max_tokens_to_sample": 3000}

llm = Bedrock(model_id="anthropic.claude-v2", model_kwargs=model_kwargs_claude)

qa = RetrievalQA.from_chain_type(
    llm=llm, retriever=retriever, return_source_documents=True
)

qa(query)

## API 参考

有关 `AmazonKnowledgeBasesRetriever` 所有功能和配置的详细文档，请访问 [API 参考](https://python.langchain.com/api_reference/aws/retrievers/langchain_aws.retrievers.bedrock.AmazonKnowledgeBasesRetriever.html)。